In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········


In [2]:
# Load the ColBERT backend — reuses the index already built in
# 08_colbert.ipynb (clean, 576 chunks)
import json
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

with open(project_root / "data" / "id_to_text.json") as f:
    id_to_text = json.load(f)

colbert_vectorstore = ColBERTVectorStoreShim(
    index_folder=str(project_root / "data" / "colbert_index"),
    id_to_text=id_to_text,
)
print(f"ColBERT index loaded: {len(id_to_text)} chunks available")

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████| 199/199 [00:00<00:00, 472.73it/s]


ColBERT index loaded: 576 chunks available


In [20]:
# Same LLM as generation.ipynb's eval loop — keeping the generation
# model IDENTICAL across every strategy isolates the retriever as the
# only variable in the comparison. NOTE: generation.ipynb's base_url
# ("http://60:45:2e:a4:cd:ba:11434") looks like a MAC address pasted
# where a host/IP belongs — fixed to a placeholder below; swap in
# your actual Ollama host so this matches generation.ipynb for real.
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
#llm = ChatOllama(
#    model="orinth-1.5:9B",
#    base_url="http://localhost:11434",  
#    temperature=0,
#)

In [8]:
# ColBERTStrategy — same .retrieve()/.run() interface every other
# strategy exposes, so it drops into the eval harness's `strategies`
# dict. Save this class into rag_lab/strategies/colbert_strategy.py;
# imported from there in the next cell for the actual run, matching
# the inline-prototype -> package-import pattern used in 02-06.
from typing import List
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


class ColBERTStrategy:
    """Naive RAG using the ColBERT/PyLate backend instead of dense
    Chroma retrieval. Same answer prompt as NaiveStrategy, so any
    quality delta measured is attributable to the retriever, not a
    different prompt."""

    def __init__(self, colbert_vectorstore: ColBERTVectorStoreShim, llm: BaseChatModel, k: int = 4):
        self.colbert_vectorstore = colbert_vectorstore
        self.retriever = colbert_vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = llm
        self.answer_prompt = ChatPromptTemplate.from_template(
            "Answer the question based only on the following context:\n"
            "{context}\n\nQuestion: {question}"
        )

    def retrieve(self, query: str) -> List[Document]:
        return self.retriever.invoke(query)

    def run(self, query: str) -> str:
        docs = self.retrieve(query)
        context = "\n\n".join(d.page_content for d in docs)
        chain = self.answer_prompt | self.llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

In [9]:
# Smoke test before the full eval loop
test_query = "Why do PINNs struggle with irregular geometry?"
_strategy = ColBERTStrategy(colbert_vectorstore, llm=llm)
_docs = _strategy.retrieve(test_query)
print(f"Retrieved {len(_docs)} docs\n")
print(_strategy.run(test_query))

Retrieved 4 docs

PINNs are built on a **global, mesh‑free collocation** strategy in which the neural network is trained by enforcing the governing PDE (and its derivatives) at a set of points that are usually taken from a regular grid or from a fairly uniform sampling of the domain.  
When the physical domain has an **irregular shape** (e.g., cracks, moving interfaces, highly non‑rectangular boundaries), two difficulties arise that make standard PINNs perform poorly:

1. **Irregular placement of collocation points** –  
   In a complex geometry the points cannot be laid out on a simple Cartesian mesh.  
   Because PINNs rely on **automatic differentiation** to evaluate spatial and temporal derivatives, the quality of those derivative estimates deteriorates when the points are unevenly spaced or clustered near sharp corners. The lack of a structured mesh also makes it hard to choose an appropriate “mesh size” or resolution, which in turn forces the user to use a very large number of po

In [10]:
# Import the committed version (save the class above into
# rag_lab/strategies/colbert_strategy.py first) and build the
# strategies dict for the eval loop — intentionally JUST "colbert"
# here: load_completed_keys below will skip every (strategy, question)
# pair the other 6 strategies already finished in generation.ipynb,
# since this appends to the SAME eval_results.jsonl.
from rag_lab.strategies.colbert_strategy import ColBERTStrategy

strategies = {
    "colbert": ColBERTStrategy(colbert_vectorstore, llm=llm),
}

In [11]:
# Eval set — same file generation.ipynb used for the other 6
# strategies, so this is a like-for-like comparison.
with open(project_root / "eval" / "eval_set.json") as f:
    eval_set = json.load(f)["questions"]
print(f"{len(eval_set)} questions loaded")

63 questions loaded


In [25]:
# Generation loop — identical shape to generation.ipynb's loop, just
# scoped to the "colbert" strategy, appending to the SAME results
# file so scoring.ipynb can compare all 7 strategies together.
#
# NOTE: pacing_delay(strategy_name) — check rag_lab/utils.py handles
# the new "colbert" key (a sensible default/fallback rather than a
# KeyError if it's a strict per-strategy lookup table).
from rag_lab.utils import call_with_backoff, is_daily_quota_error, load_completed_keys, append_jsonl, pacing_delay
import time

RESULTS_PATH = project_root / "eval" / "eval_results.jsonl"
completed = load_completed_keys(RESULTS_PATH)
print(f"{len(completed)} (strategy, question) pairs already done\n")

quota_hit = False
for strategy_name, strategy in strategies.items():
    if quota_hit:
        break
    for item in eval_set:
        key = (strategy_name, item["id"])
        if key in completed:
            continue
        try:
            contexts = call_with_backoff(lambda: strategy.retrieve(item["question"]))
            answer = call_with_backoff(lambda: strategy.run(item["question"]))
            append_jsonl(RESULTS_PATH, {
                "strategy": strategy_name, "id": item["id"],
                "question": item["question"], "question_type": item["question_type"],
                "contexts": [d.page_content for d in contexts],
                "answer": answer, "ground_truth": item.get("ground_truth", ""),
            })
            completed.add(key)
            print(f"  [{strategy_name}] {item['id']} done")
        except Exception as e:
            if is_daily_quota_error(e):
                print(f"\nDAILY QUOTA EXHAUSTED at [{strategy_name}] {item['id']}. Resume later — checkpoint saved.\n")
                quota_hit = True
                break
            print(f"  [{strategy_name}] {item['id']} FAILED: {e}")
        time.sleep(pacing_delay(strategy_name))
    if not quota_hit:
        print(f"=== Finished {strategy_name} ===\n")

440 (strategy, question) pairs already done

  [colbert] b3_cross_008 done
=== Finished colbert ===



In [26]:
# Quick sanity check: confirm results for "colbert" actually landed
# alongside the other 6 strategies in the same file.
from collections import Counter
with open(RESULTS_PATH) as f:
    rows = [json.loads(line) for line in f]
print(Counter(r["strategy"] for r in rows))

Counter({'naive': 63, 'multi_query': 63, 'rag_fusion': 63, 'hyde': 63, 'step_back': 63, 'decomposition': 63, 'colbert': 63})
